## Installing Required Libraries

In [1]:
!pip install ultralytics opencv-python filterpy lap


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.6 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=63090afa8640f7fb960cd9904fa54c7a935e1c79007e1eda1ad314cfeffe6169
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


## Uploading a Video File

In [2]:
from google.colab import files
uploaded = files.upload()


Saving mixkit-people-in-the-subway-hall-in-tokyo-4454-hd-ready.mp4 to mixkit-people-in-the-subway-hall-in-tokyo-4454-hd-ready.mp4


## Importing Libraries

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## SORT Tracker Code

In [4]:
from filterpy.kalman import KalmanFilter

class Sort:
    def __init__(self):
        self.trackers = []
        self.next_id = 0

    def update(self, detections):
        results = []
        for det in detections:
            x1, y1, x2, y2, score = det
            results.append([x1, y1, x2, y2, self.next_id])
            self.next_id += 1
        return results


## Loading YOLO Pretrained Model

In [5]:
model = YOLO("yolov8n.pt")  # lightweight pretrained model
tracker = Sort()


## Object Detection + Tracking on Video

In [6]:
video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    detections = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append([x1, y1, x2, y2, conf])

    tracks = tracker.update(detections)

    for x1, y1, x2, y2, track_id in tracks:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {track_id}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

    cv2.imshow("Object Detection & Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


Display

In [8]:
from google.colab.patches import cv2_imshow

cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    detections = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append([x1, y1, x2, y2, conf])

    tracks = tracker.update(detections)

    for x1, y1, x2, y2, track_id in tracks:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {track_id}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

    cv2_imshow(frame)



## Showing Result Frame by Frame and downloading the video

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

cap = cv2.VideoCapture("mixkit-people-in-the-subway-hall-in-tokyo-4454-hd-ready.mp4")
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    detections = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append([x1, y1, x2, y2, conf])

    tracks = tracker.update(detections)

    for x1, y1, x2, y2, track_id in tracks:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {track_id}",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255,0,0), 2)

    frame_count += 1

    # Show every 10th frame (prevents flooding)
    if frame_count % 10 == 0:
        cv2_imshow(frame)

cap.release()


In [10]:
from google.colab.patches import cv2_imshow
import cv2

cap = cv2.VideoCapture("mixkit-people-in-the-subway-hall-in-tokyo-4454-hd-ready.mp4")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    "output_tracking.mp4",
    fourcc,
    20,
    (int(cap.get(3)), int(cap.get(4)))
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    detections = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append([x1, y1, x2, y2, conf])

    tracks = tracker.update(detections)

    for x1, y1, x2, y2, track_id in tracks:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {track_id}",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255,0,0), 2)

    out.write(frame)

cap.release()
out.release()
